#Logic Strategy

1. Thông tin cơ bản
- Loại chiến lược: Mean Reversion
- Position: Long Only
- Asset & Timeframe: BNB USDT - Khung 1h
2. Entry Logic
- EMA100 > EMA50
- RSI < 40
- RSI(t) - RSI(t-1) <= 3
3. Exit Rules
- TP: RSI > 65 AND EMA9 > EMA50
- SL: RSI <= 20

## Python Code

In [ ]:
# Install required libraries
!pip install pandas pandas-ta requests plotly -q

import requests
import time
import pandas as pd
import pandas_ta as ta
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

# GLOBAL CONFIGURATION
CONFIG = {
    'SYMBOLS': ['BTCUSDT', 'BNBUSDT'],
    'INTERVAL': '1h',
    'START_DATE': '01/04/2020',
    'END_DATE': '30/04/2021',
    'INITIAL_CAPITAL': 10000.0,
    'POSITION_SIZE_PCT': 0.10,
    'FEE_RATE': 0.0004,
    'SLIPPAGE': 0.0002
}

# MODULE 1: DATA HANDLER
class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.symbol = symbol.upper()
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return int(dt.timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"
        req_symbol = self.symbol

        print(f"Checking connection for {req_symbol}...")
        try:
            test = requests.get(f"{base_url}/api/v3/time", timeout=5)
            if test.status_code == 451:
                print("US IP detected. Switching to Binance.US...")
                base_url = "https://api.binance.us"
                if req_symbol == "ETHUSDT": req_symbol = "ETHUSD"
                if req_symbol == "BNBUSDT": req_symbol = "BNBUSD"
        except Exception:
            base_url = "https://api.binance.us"
            if req_symbol == "ETHUSDT": req_symbol = "ETHUSD"
            if req_symbol == "BNBUSDT": req_symbol = "BNBUSD"

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)

        if self.end_date == datetime.now().strftime("%d/%m/%Y"):
            end_time = int(time.time() * 1000)
        else:
            end_dt = datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
            end_time = int(end_dt.timestamp() * 1000)

        all_klines =[]
        print(f"Downloading {req_symbol} ({self.interval})...")

        while start_time < end_time:
            params = {"symbol": req_symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except Exception: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        df = df.astype(float)
        print(f"Total: {len(df)} candles fetched.")
        return df

# MODULE 2: STRATEGY LOGIC
class MeanReversionStrategy:
    def __init__(self):
        self.rsi_period = 14
        self.ema_9 = 9
        self.ema_50 = 50
        self.ema_100 = 100
        self.rsi_entry_max = 40
        self.rsi_drop_val = 3.0
        self.rsi_tp_level = 65
        self.rsi_sl_level = 20

    def calculate_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        df['RSI'] = ta.rsi(df['Close'], length=self.rsi_period)
        df['EMA_9'] = ta.ema(df['Close'], length=self.ema_9)
        df['EMA_50'] = ta.ema(df['Close'], length=self.ema_50)
        df['EMA_100'] = ta.ema(df['Close'], length=self.ema_100)
        df.dropna(inplace=True)
        return df

# MODULE 3: BACKTEST ENGINE (LONG ONLY DIP BUYING)
class BacktestEngine:
    def __init__(self, df, strategy, config):
        self.df = strategy.calculate_indicators(df.copy())
        self.strategy = strategy
        self.config = config

        self.capital = config['INITIAL_CAPITAL']
        self.invested_amount = 0.0
        self.position = 0
        self.entry_price = 0.0
        self.trades =[]

        # THAY ĐỔI: Sử dụng Numpy array để lưu trữ equity mượt mà hơn
        self.equity_curve = np.zeros(len(self.df))

    def run(self):
        O = self.df['Open'].values
        C = self.df['Close'].values
        times = self.df.index

        rsi = self.df['RSI'].values
        ema9 = self.df['EMA_9'].values
        ema50 = self.df['EMA_50'].values
        ema100 = self.df['EMA_100'].values

        fee = self.config['FEE_RATE']
        slip = self.config['SLIPPAGE']
        pos_size_pct = self.config['POSITION_SIZE_PCT']

        s = self.strategy

        # THAY ĐỔI: Khởi tạo vốn cho 2 nến đầu tiên
        self.equity_curve[0] = self.capital
        self.equity_curve[1] = self.capital

        for i in range(2, len(self.df)):
            # EXIT LOGIC
            if self.position == 1:
                exit_signal = False
                reason = ""

                if rsi[i-1] <= s.rsi_sl_level:
                    exit_signal = True
                    reason = "SL (RSI <= 20)"

                elif (rsi[i-1] > s.rsi_tp_level) and (ema9[i-1] > ema50[i-1]):
                    exit_signal = True
                    reason = "TP (RSI > 65 and EMA Cross)"

                if exit_signal:
                    real_exit = O[i] * (1 - slip)
                    pnl_pct = (real_exit - self.entry_price) / self.entry_price

                    net_pnl_pct = pnl_pct - (fee * 2)
                    pnl_amt = self.invested_amount * net_pnl_pct
                    self.capital += pnl_amt

                    self.trades.append({
                        'Date': times[i], 'Type': 'Long',
                        'Reason': reason,
                        'Entry': self.entry_price, 'Exit': real_exit,
                        'PnL_Amt': pnl_amt, 'Equity': self.capital
                    })

                    self.position = 0
                    self.invested_amount = 0.0

            # ENTRY LOGIC
            if self.position == 0:
                rsi_0 = rsi[i-1]
                rsi_minus_1 = rsi[i-2]

                cond_1 = rsi_0 < s.rsi_entry_max
                cond_2 = rsi_minus_1 >= (rsi_0 + s.rsi_drop_val)
                cond_3 = ema100[i-1] > ema50[i-1]

                if cond_1 and cond_2 and cond_3:
                    self.position = 1
                    self.entry_price = O[i] * (1 + slip)
                    self.invested_amount = self.capital * pos_size_pct

            # CAL UNREALIZED PNL
            current_equity = self.capital
            if self.position == 1:
                current_value = self.invested_amount * (C[i] / self.entry_price)
                unrealized_pnl = current_value - self.invested_amount
                current_equity += unrealized_pnl

            self.equity_curve[i] = current_equity

        return pd.Series(self.equity_curve, index=self.df.index), pd.DataFrame(self.trades)

# MODULE 4: PERFORMANCE EVALUATOR
class Evaluator:
    @staticmethod
    def generate_report(equity, trades_df, initial_capital, symbol, interval, pos_size_pct):
        if trades_df.empty:
            print(f"No trades executed for {symbol}!")
            return

        final_equity = equity.iloc[-1]
        net_pnl = final_equity - initial_capital
        total_return = net_pnl / initial_capital

        rolling_max = equity.cummax()
        drawdown = (equity - rolling_max) / rolling_max
        max_dd = drawdown.min()

        wins = trades_df[trades_df['PnL_Amt'] > 0]
        losses = trades_df[trades_df['PnL_Amt'] <= 0]
        winrate = len(wins) / len(trades_df) if len(trades_df) > 0 else 0
        profit_factor = wins['PnL_Amt'].sum() / abs(losses['PnL_Amt'].sum()) if len(losses) > 0 and losses['PnL_Amt'].sum() != 0 else 0

        days = (equity.index[-1] - equity.index[0]).days
        cagr = ((final_equity / initial_capital) ** (365.0 / days)) - 1 if days > 0 else 0

        returns = equity.pct_change().dropna()
        sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(365 * 24) if returns.std() != 0 else 0

        print("")
        print(f"========== PERFORMANCE REPORT: {symbol} ({interval}) ==========")
        print(f"{'Strategy:':>20} Mean Reversion (Buy the Dip)")
        print(f"{'Position Sizing:':>20} {pos_size_pct*100}% of Capital")
        print(f"{'Initial Capital:':>20} ${initial_capital:,.2f}")
        print(f"{'Final Equity:':>20} ${final_equity:,.2f}")
        print(f"{'Net PnL:':>20} ${net_pnl:,.2f}")
        print(f"{'Total Return:':>20} {total_return:.2%}")
        print(f"{'CAGR (Annual):':>20} {cagr:.2%}")
        print(f"{'Max Drawdown:':>20} {max_dd:.2%}")
        print(f"{'Sharpe Ratio:':>20} {sharpe_ratio:.2f}")
        print(f"{'Total Trades:':>20} {len(trades_df)}")
        print(f"{'Winrate:':>20} {winrate:.2%}")
        print(f"{'Profit Factor:':>20} {profit_factor:.2f}")
        print("====================================================\n")

        # Plotly Implementation - Light/White Theme with Subplots for Drawdown
        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.08,
            row_heights=[0.7, 0.3]
        )

        # Trace 1: Equity Curve
        fig.add_trace(go.Scatter(
            x=equity.index, y=equity.values,
            mode="lines", name="Equity",
            line=dict(color="#2962ff", width=1.5)
        ), row=1, col=1)

        # Trace 2: Drawdown Area
        fig.add_trace(go.Scatter(
            x=drawdown.index, y=drawdown.values,
            mode="lines", name="Drawdown",
            line=dict(color="#d32f2f", width=1),
            fill='tozeroy', fillcolor='rgba(211, 47, 47, 0.2)'
        ), row=2, col=1)

        fig.update_layout(
            title=f"Portfolio Performance: {symbol} ({interval})",
            template="plotly_white",
            height=650,
            margin=dict(l=40, r=40, t=60, b=40),
            showlegend=False
        )

        fig.update_xaxes(showgrid=True, gridcolor='#e6e6e6', zeroline=False)
        fig.update_yaxes(title_text="Balance ($)", tickprefix="$", showgrid=True, gridcolor='#e6e6e6', zeroline=False, row=1, col=1)
        fig.update_yaxes(title_text="Drawdown %", tickformat=".1%", showgrid=True, gridcolor='#e6e6e6', zeroline=True, zerolinecolor='#cccccc', row=2, col=1)

        fig.show()

# MAIN EXECUTION
if __name__ == "__main__":
    # Loop through multiple symbols
    for symbol in CONFIG['SYMBOLS']:
        print(f"\n---> Starting Backtest for: {symbol} <---")
        fetcher = BinanceDataFetcher(symbol, CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])
        df_raw = fetcher.fetch()

        if not df_raw.empty:
            my_strategy = MeanReversionStrategy()
            engine = BacktestEngine(df_raw, my_strategy, CONFIG)
            equity, trades = engine.run()

            Evaluator.generate_report(
                equity,
                trades,
                CONFIG['INITIAL_CAPITAL'],
                symbol,
                CONFIG['INTERVAL'],
                CONFIG['POSITION_SIZE_PCT']
            )


---> Starting Backtest for: BTCUSDT <---
Checking connection for BTCUSDT...
Total: 9462 candles fetched.

========== PERFORMANCE REPORT: BTCUSDT (1h) ==========
           Strategy: Mean Reversion (Buy the Dip)
    Position Sizing: 10.0% of Capital
    Initial Capital: $10,000.00
       Final Equity: $10,472.65
            Net PnL: $472.65
       Total Return: 4.73%
      CAGR (Annual): 4.42%
       Max Drawdown: -2.38%
       Sharpe Ratio: 1.10
       Total Trades: 41
            Winrate: 68.29%
      Profit Factor: 2.26




---> Starting Backtest for: BNBUSDT <---
Checking connection for BNBUSDT...
Total: 9462 candles fetched.

========== PERFORMANCE REPORT: BNBUSDT (1h) ==========
           Strategy: Mean Reversion (Buy the Dip)
    Position Sizing: 10.0% of Capital
    Initial Capital: $10,000.00
       Final Equity: $11,645.35
            Net PnL: $1,645.35
       Total Return: 16.45%
      CAGR (Annual): 15.32%
       Max Drawdown: -2.69%
       Sharpe Ratio: 2.81
       Total Trades: 45
            Winrate: 77.78%
      Profit Factor: 4.51



## Go Code

In [ ]:
package main

import (
	"encoding/json"
	"fmt"
	"io/ioutil"
	"math"
	"net/http"
	"strconv"
	"time"
)

// GLOBAL CONFIG
const (
	Symbol           = "BNBUSDT"
	Interval         = "1h"
	StartDate        = "01/01/2021"
	EndDate          = "31/12/2025"
	InitialCapital   = 10000.0
	PositionSizePct  = 0.10
	FeeRate          = 0.0004
	Slippage         = 0.0002
)

type Candle struct {
	Timestamp int64
	Open      float64
	High      float64
	Low       float64
	Close     float64
}

type Trade struct {
	Type     string
	Reason   string
	Entry    float64
	Exit     float64
	PnLAmt   float64
	Equity   float64
}

func parseDate(dateStr string) int64 {
	t, _ := time.Parse("02/01/2006", dateStr)
	return t.Unix() * 1000
}

func fetchBinanceData()[]Candle {
	baseURL := "https://data-api.binance.vision"
	startTime := parseDate(StartDate)

	endT, _ := time.Parse("02/01/2006", EndDate)
	endTime := endT.Add(23*time.Hour + 59*time.Minute).Unix() * 1000

	var allCandles[]Candle
	fmt.Printf("Downloading %s (%s)...\n", Symbol, Interval)

	for startTime < endTime {
		url := fmt.Sprintf("%s/api/v3/klines?symbol=%s&interval=%s&startTime=%d&endTime=%d&limit=1000",
			baseURL, Symbol, Interval, startTime, endTime)

		resp, err := http.Get(url)
		if err != nil || resp.StatusCode != 200 {
			break
		}

		body, _ := ioutil.ReadAll(resp.Body)
		var result [][]interface{}
		json.Unmarshal(body, &result)

		if len(result) == 0 {
			break
		}

		for _, item := range result {
			ts := int64(item[0].(float64))
			open, _ := strconv.ParseFloat(item[1].(string), 64)
			high, _ := strconv.ParseFloat(item[2].(string), 64)
			low, _ := strconv.ParseFloat(item[3].(string), 64)
			close, _ := strconv.ParseFloat(item[4].(string), 64)

			allCandles = append(allCandles, Candle{Timestamp: ts, Open: open, High: high, Low: low, Close: close})
		}

		lastCandle := result[len(result)-1]
		startTime = int64(lastCandle[0].(float64)) + 1
	}
	fmt.Printf("Total: %d candles fetched.\n", len(allCandles))
	return allCandles
}

func calculateEMA(closes []float64, period int)[]float64 {
	ema := make([]float64, len(closes))
	if len(closes) < period {
		return ema
	}

	multiplier := 2.0 / float64(period+1)
	sum := 0.0
	for i := 0; i < period; i++ {
		sum += closes[i]
	}
	ema[period-1] = sum / float64(period)

	for i := period; i < len(closes); i++ {
		ema[i] = (closes[i]-ema[i-1])*multiplier + ema[i-1]
	}
	return ema
}

func calculateRSI(closes[]float64, period int) []float64 {
	rsi := make([]float64, len(closes))
	if len(closes) < period+1 {
		return rsi
	}

	gains := make([]float64, len(closes))
	losses := make([]float64, len(closes))

	for i := 1; i < len(closes); i++ {
		change := closes[i] - closes[i-1]
		if change > 0 {
			gains[i] = change
		} else {
			losses[i] = -change
		}
	}

	avgGain := 0.0
	avgLoss := 0.0
	for i := 1; i <= period; i++ {
		avgGain += gains[i]
		avgLoss += losses[i]
	}
	avgGain /= float64(period)
	avgLoss /= float64(period)

	rsi[period] = 100.0
	if avgLoss != 0 {
		rs := avgGain / avgLoss
		rsi[period] = 100.0 - (100.0 / (1.0 + rs))
	}

	for i := period + 1; i < len(closes); i++ {
		avgGain = (avgGain*float64(period-1) + gains[i]) / float64(period)
		avgLoss = (avgLoss*float64(period-1) + losses[i]) / float64(period)

		if avgLoss == 0 {
			rsi[i] = 100.0
		} else {
			rs := avgGain / avgLoss
			rsi[i] = 100.0 - (100.0 / (1.0 + rs))
		}
	}
	return rsi
}

func main() {
	candles := fetchBinanceData()
	if len(candles) == 0 {
		fmt.Println("No data fetched.")
		return
	}

	closes := make([]float64, len(candles))
	opens := make([]float64, len(candles))
	for i, c := range candles {
		closes[i] = c.Close
		opens[i] = c.Open
	}

	ema9 := calculateEMA(closes, 9)
	ema50 := calculateEMA(closes, 50)
	ema100 := calculateEMA(closes, 100)
	rsi := calculateRSI(closes, 14)

	capital := InitialCapital
	position := 0
	entryPrice := 0.0
	investedAmount := 0.0
	var trades[]Trade

	highestCapital := capital
	maxDrawdown := 0.0

	fmt.Println("Running Backtest Engine (Mean Reversion)...")

	for i := 2; i < len(candles); i++ {
		if position == 1 {
			exitSignal := false
			reason := ""

			if rsi[i-1] <= 20 {
				exitSignal = true
				reason = "SL"
			} else if rsi[i-1] > 65 && ema9[i-1] > ema50[i-1] {
				exitSignal = true
				reason = "TP"
			}

			if exitSignal {
				realExit := opens[i] * (1 - Slippage)
				pnlPct := (realExit - entryPrice) / entryPrice
				netPnlPct := pnlPct - (FeeRate * 2)
				pnlAmt := investedAmount * netPnlPct
				capital += pnlAmt

				trades = append(trades, Trade{
					Type:   "Long",
					Reason: reason,
					Entry:  entryPrice,
					Exit:   realExit,
					PnLAmt: pnlAmt,
					Equity: capital,
				})

				position = 0
				investedAmount = 0.0
			}
		}

		if position == 0 {
			rsi0 := rsi[i-1]
			rsiMinus1 := rsi[i-2]

			cond1 := rsi0 < 40
			cond2 := rsiMinus1 >= (rsi0 + 3.0)
			cond3 := ema100[i-1] > ema50[i-1]

			if cond1 && cond2 && cond3 {
				position = 1
				entryPrice = opens[i] * (1 + Slippage)
				investedAmount = capital * PositionSizePct
			}
		}

		if capital > highestCapital {
			highestCapital = capital
		}
		dd := (capital - highestCapital) / highestCapital
		if dd < maxDrawdown {
			maxDrawdown = dd
		}
	}

	// Calculate metrics
	netPnL := capital - InitialCapital
	totalReturn := netPnL / InitialCapital

	wins := 0
	grossProfit := 0.0
	grossLoss := 0.0
	for _, t := range trades {
		if t.PnLAmt > 0 {
			wins++
			grossProfit += t.PnLAmt
		} else {
			grossLoss += math.Abs(t.PnLAmt)
		}
	}

	winrate := 0.0
	profitFactor := 0.0
	if len(trades) > 0 {
		winrate = float64(wins) / float64(len(trades))
	}
	if grossLoss > 0 {
		profitFactor = grossProfit / grossLoss
	}

	days := float64(len(candles)) / 24.0
	cagr := 0.0
	if days > 0 {
		cagr = math.Pow(capital/InitialCapital, 365.0/days) - 1
	}

	fmt.Println("\nPERFORMANCE REPORT")
	fmt.Printf("Strategy:             Mean Reversion (Buy the Dip)\n")
	fmt.Printf("Initial Capital:      $%.2f\n", InitialCapital)
	fmt.Printf("Final Equity:         $%.2f\n", capital)
	fmt.Printf("Net PnL:              $%.2f\n", netPnL)
	fmt.Printf("Total Return:         %.2f%%\n", totalReturn*100)
	fmt.Printf("CAGR (Annual):        %.2f%%\n", cagr*100)
	fmt.Printf("Max Drawdown:         %.2f%%\n", maxDrawdown*100)
	fmt.Printf("Total Trades:         %d\n", len(trades))
	fmt.Printf("Winrate:              %.2f%%\n", winrate*100)
	fmt.Printf("Profit Factor:        %.2f\n", profitFactor)
	fmt.Println("")
}

# Multi asset/timeframe

In [ ]:
# ==============================================================================
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN & IMPORT
# ==============================================================================
!pip install pandas_ta -q

import requests
import time
import pandas as pd
import pandas_ta as ta
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# BƯỚC 2: CẤU HÌNH HỆ THỐNG (MULTI-ASSET & MULTI-TIMEFRAME)
# ==============================================================================
CONFIG = {
    # Khai báo danh sách tài sản và khung thời gian muốn test
    'SYMBOLS': ['BTCUSDT', 'ETHUSDT', 'BNBUSDT'],
    'TIMEFRAMES': ['15m', '1h'],

    'START_DATE': '01/01/2021',
    'END_DATE': '31/12/2022',
    'INITIAL_CAPITAL': 10000.0,

    'POSITION_SIZE_PCT': 0.10,   # Đánh 10% vốn mỗi lệnh
    'FEE_RATE': 0.0004,          # Phí giao dịch 0.04%
    'SLIPPAGE': 0.0002           # Trượt giá 0.02%
}

# ==============================================================================
# MODULE 1: DATA HANDLER
# ==============================================================================
class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.symbol = symbol.upper()
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return int(dt.timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"
        req_symbol = self.symbol

        try:
            test = requests.get(f"{base_url}/api/v3/time", timeout=5)
            if test.status_code == 451:
                base_url = "https://api.binance.us"
                if req_symbol in ["BTCUSDT", "ETHUSDT", "BNBUSDT"]:
                    req_symbol = req_symbol.replace("USDT", "USD")
        except Exception:
            base_url = "https://api.binance.us"
            if req_symbol in ["BTCUSDT", "ETHUSDT", "BNBUSDT"]:
                req_symbol = req_symbol.replace("USDT", "USD")

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)

        if self.end_date == datetime.now().strftime("%d/%m/%Y"):
            end_time = int(time.time() * 1000)
        else:
            end_dt = datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
            end_time = int(end_dt.timestamp() * 1000)

        all_klines = []
        print(f"⏳ Downloading {req_symbol} ({self.interval})...", end=" ")

        while start_time < end_time:
            params = {"symbol": req_symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except Exception: break

        if not all_klines:
            print("Failed.")
            return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        df = df.astype(float)
        print(f"✅ {len(df)} candles.")
        return df

# ==============================================================================
# MODULE 2: STRATEGY (LOGIC MEAN REVERSION)
# ==============================================================================
class MeanReversionStrategy:
    def __init__(self):
        self.rsi_period = 14
        self.ema_9 = 9
        self.ema_50 = 50
        self.ema_100 = 100
        self.rsi_entry_max = 40
        self.rsi_drop_val = 3.0
        self.rsi_tp_level = 65
        self.rsi_sl_level = 20

    def calculate_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        df['RSI'] = ta.rsi(df['Close'], length=self.rsi_period)
        df['EMA_9'] = ta.ema(df['Close'], length=self.ema_9)
        df['EMA_50'] = ta.ema(df['Close'], length=self.ema_50)
        df['EMA_100'] = ta.ema(df['Close'], length=self.ema_100)
        df.dropna(inplace=True)
        return df

# ==============================================================================
# MODULE 3: BACKTEST ENGINE
# ==============================================================================
class BacktestEngine:
    def __init__(self, df, strategy, config):
        self.df = strategy.calculate_indicators(df.copy())
        self.strategy = strategy
        self.config = config
        self.capital = config['INITIAL_CAPITAL']
        self.invested_amount = 0.0
        self.position = 0
        self.entry_price = 0.0
        self.trades = []
        self.equity_curve = [self.capital] * len(self.df)

    def run(self):
        O = self.df['Open'].values; C = self.df['Close'].values
        times = self.df.index
        rsi = self.df['RSI'].values; ema9 = self.df['EMA_9'].values
        ema50 = self.df['EMA_50'].values; ema100 = self.df['EMA_100'].values

        fee = self.config['FEE_RATE']; slip = self.config['SLIPPAGE']
        pos_size_pct = self.config['POSITION_SIZE_PCT']
        s = self.strategy

        for i in range(2, len(self.df)):
            # --- EXIT LOGIC ---
            if self.position == 1:
                exit_signal = False
                reason = ""

                if rsi[i-1] <= s.rsi_sl_level:
                    exit_signal = True; reason = "SL"
                elif (rsi[i-1] > s.rsi_tp_level) or (ema9[i-1] > ema50[i-1]):
                    exit_signal = True; reason = "TP"

                if exit_signal:
                    real_exit = O[i] * (1 - slip)
                    pnl_pct = (real_exit - self.entry_price) / self.entry_price
                    net_pnl_pct = pnl_pct - (fee * 2)
                    pnl_amt = self.invested_amount * net_pnl_pct
                    self.capital += pnl_amt

                    self.trades.append({
                        'Date': times[i], 'Type': 'Long', 'Reason': reason,
                        'Entry': self.entry_price, 'Exit': real_exit,
                        'PnL_Amt': pnl_amt, 'Equity': self.capital
                    })
                    self.position = 0; self.invested_amount = 0.0

            # --- ENTRY LOGIC ---
            if self.position == 0:
                cond_1 = rsi[i-1] < s.rsi_entry_max
                cond_2 = rsi[i-2] >= (rsi[i-1] + s.rsi_drop_val)
                cond_3 = ema100[i-1] > ema50[i-1]

                if cond_1 and cond_2 and cond_3:
                    self.position = 1
                    self.entry_price = O[i] * (1 + slip)
                    self.invested_amount = self.capital * pos_size_pct

            self.equity_curve[i] = self.capital

        return pd.Series(self.equity_curve, index=self.df.index), pd.DataFrame(self.trades)

# ==============================================================================
# MODULE 4: PERFORMANCE EVALUATOR (RETURNS STATS FOR AGGREGATION)
# ==============================================================================
class Evaluator:
    @staticmethod
    def get_metrics(equity, trades_df, initial_capital, symbol, timeframe):
        if trades_df.empty:
            return {
                'Symbol': symbol, 'TF': timeframe, 'Net PnL': 0, 'Return %': 0,
                'Max DD %': 0, 'Trades': 0, 'Winrate %': 0, 'Profit Factor': 0
            }

        final_equity = equity.iloc[-1]
        net_pnl = final_equity - initial_capital
        total_return = (net_pnl / initial_capital) * 100

        rolling_max = equity.cummax()
        drawdown = (equity - rolling_max) / rolling_max
        max_dd = drawdown.min() * 100

        wins = trades_df[trades_df['PnL_Amt'] > 0]
        losses = trades_df[trades_df['PnL_Amt'] <= 0]
        winrate = (len(wins) / len(trades_df)) * 100
        profit_factor = wins['PnL_Amt'].sum() / abs(losses['PnL_Amt'].sum()) if len(losses) > 0 and losses['PnL_Amt'].sum() != 0 else 0

        return {
            'Symbol': symbol,
            'TF': timeframe,
            'Net PnL': round(net_pnl, 2),
            'Return %': round(total_return, 2),
            'Max DD %': round(max_dd, 2),
            'Trades': len(trades_df),
            'Winrate %': round(winrate, 2),
            'Profit Factor': round(profit_factor, 2)
        }

# ==============================================================================
# MAIN BATCH EXECUTION
# ==============================================================================
if __name__ == "__main__":
    print(f"🚀 BẮT ĐẦU BACKTEST MULTI-ASSET & MULTI-TIMEFRAME")
    print("="*60)

    all_results = []

    for symbol in CONFIG['SYMBOLS']:
        for tf in CONFIG['TIMEFRAMES']:
            fetcher = BinanceDataFetcher(symbol, tf, CONFIG['START_DATE'], CONFIG['END_DATE'])
            df_raw = fetcher.fetch()

            if not df_raw.empty:
                my_strategy = MeanReversionStrategy()
                engine = BacktestEngine(df_raw, my_strategy, CONFIG)
                equity, trades = engine.run()

                # Tính toán và lưu chỉ số vào mảng kết quả
                metrics = Evaluator.get_metrics(equity, trades, CONFIG['INITIAL_CAPITAL'], symbol, tf)
                all_results.append(metrics)

            # Tạm nghỉ 1 chút để tránh bị Binance chặn IP do gọi API quá nhanh
            time.sleep(1)

    # Hiển thị bảng tổng hợp kết quả
    print("\n" + "="*80)
    print("📊 BẢNG TỔNG HỢP KẾT QUẢ BACKTEST")
    print("="*80)
    if all_results:
        results_df = pd.DataFrame(all_results)
        # Sắp xếp theo Return % giảm dần để xem tổ hợp nào ngon nhất
        results_df = results_df.sort_values(by='Return %', ascending=False).reset_index(drop=True)
        print(results_df.to_string(index=False))
    else:
        print("⚠️ Không có dữ liệu để tổng hợp.")
    print("="*80)

🚀 BẮT ĐẦU BACKTEST MULTI-ASSET & MULTI-TIMEFRAME
⏳ Downloading BTCUSDT (15m)... ✅ 70015 candles.
⏳ Downloading BTCUSDT (1h)... ✅ 17507 candles.
⏳ Downloading ETHUSDT (15m)... ✅ 70015 candles.
⏳ Downloading ETHUSDT (1h)... ✅ 17507 candles.
⏳ Downloading BNBUSDT (15m)... ✅ 70015 candles.
⏳ Downloading BNBUSDT (1h)... ✅ 17507 candles.

📊 BẢNG TỔNG HỢP KẾT QUẢ BACKTEST
 Symbol  TF  Net PnL  Return %  Max DD %  Trades  Winrate %  Profit Factor
BNBUSDT  1h   186.82      1.87     -7.73     154      59.09           1.06
BTCUSDT  1h    78.24      0.78     -4.98     184      59.78           1.03
ETHUSDT  1h  -457.82     -4.58     -9.09     154      56.49           0.86
BNBUSDT 15m  -767.52     -7.68    -15.97     624      56.25           0.89
BTCUSDT 15m  -835.83     -8.36     -8.99     613      58.08           0.85
ETHUSDT 15m -1356.20    -13.56    -16.26     604      55.13           0.81
